# 🛡️ Comment Guard - Model Training (GPU)

This notebook trains the **toxic-bert** model for Telugu-English code-mixed toxicity detection.

### ⚡ IMPORTANT: Enable GPU First!
1. Go to **Runtime → Change runtime type**
2. Set **Hardware accelerator** to **T4 GPU**
3. Click **Save**

## Step 0: Mount Google Drive (Recommended)

Mounting Google Drive allows you to save your model checkpoints permanently. If Colab disconnects, you can resume from where you left off.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Create a dedicated folder for Comment Guard in your Drive
DRIVE_PATH = '/content/drive/MyDrive/CommentGuard_Model'
os.makedirs(DRIVE_PATH, exist_ok=True)

print(f"✅ Google Drive mounted! Model will be saved to: {DRIVE_PATH}")

## Step 1: Install Dependencies

In [ ]:
!pip install transformers torch scikit-learn accelerate openpyxl pandas -q

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU not detected! Go to Runtime → Change runtime type → T4 GPU")

## Step 2: Upload Your Data Files

Run the cell below and upload these files from your `backend/data/` folder:
1. `training_data_telugu-hate.xlsx` (REQUIRED)
2. `telugu_badwords.txt` (optional but recommended)
3. `secure_words.bin` (optional but recommended)
4. `bad_emojis.txt` (optional but recommended)

In [ ]:
import os
from google.colab import files

# Create data directory
os.makedirs('data', exist_ok=True)

print("📁 Upload your data files (training_data_telugu-hate.xlsx, telugu_badwords.txt, secure_words.bin, bad_emojis.txt)")
print("You can select ALL 4 files at once!\n")

uploaded = files.upload()

for filename in uploaded:
    with open(f'data/{filename}', 'wb') as f:
        f.write(uploaded[filename])
    print(f"  ✅ Saved: data/{filename}")

print(f"\n📂 Files in data/: {os.listdir('data')}")

## Step 3: Train the Model

This cell contains the full training pipeline. On a T4 GPU, training should complete in **5-10 minutes** (vs 1-2 hours on CPU).

In [ ]:
import os
import sys
import json
import base64
import random
from pathlib import Path

import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from torch.utils.data import Dataset as TorchDataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

# ── Config ────────────────────────────────────────────────────────────────────
DATA_DIR   = Path('data')
if os.path.exists('/content/drive/MyDrive'):
    OUTPUT_DIR = Path('/content/drive/MyDrive/CommentGuard_Model')
    print(f"📂 Weights will be saved to Google Drive: {OUTPUT_DIR}")
else:
    OUTPUT_DIR = Path('model_output')
    print("⚠️ Google Drive not mounted. Weights will be saved locally (and lost on disconnect).")

BASE_MODEL = 'unitary/toxic-bert'
MAX_LENGTH = 128
EPOCHS     = 8
LEARNING_RATE = 3e-5

random.seed(42)

# ── Helper Functions ──────────────────────────────────────────────────────────
def is_code_mixed(text):
    text = str(text)
    has_latin = any('\u0041' <= c <= '\u007A' for c in text)
    total = len([c for c in text if c.strip()])
    telugu = len([c for c in text if '\u0C00' <= c <= '\u0C7F'])
    if total == 0 or not has_latin:
        return False
    if telugu / total > 0.8:
        return False
    return True

def load_data(files_list):
    hate_labels_set = {'hate', 'offensive', 'hof', '1', 'yes', 'toxic'}
    frames = []
    TEXT_NAMES  = {'text', 'comment', 'comments', 'sentence', 'tweet', 'content', 'data'}
    LABEL_NAMES = {'label', 'labels', 'category', 'class', 'tag', 'hate', 'annotation'}

    for excel_file in files_list:
        print(f"  Loading: {excel_file.name}")
        try:
            if excel_file.suffix == '.csv':
                sheets_data = [('csv', pd.read_csv(excel_file))]
            else:
                xl = pd.ExcelFile(excel_file)
                sheets_data = [(sheet, xl.parse(sheet)) for sheet in xl.sheet_names]

            for sheet, df in sheets_data:
                text_col = next(
                    (c for c in df.columns if str(c).lower() in TEXT_NAMES or
                     any(t in str(c).lower() for t in ['text', 'comment', 'sentence'])), None)
                label_col = next(
                    (c for c in df.columns if str(c).lower() in LABEL_NAMES or
                     any(t in str(c).lower() for t in ['label', 'categor', 'class'])), None)

                if text_col and str(text_col).lower() in ['s.no', 'no', 'id', 'index', 'sr']:
                    text_col = None

                if text_col and label_col:
                    sub = df[[text_col, label_col]].copy()
                    sub.columns = ['text', 'label']
                    sub = sub.dropna()
                    sub['label'] = sub['label'].astype(str).str.strip().str.lower()
                    sub['label_int'] = sub['label'].apply(lambda x: 1 if x in hate_labels_set else 0)
                    before = len(sub)
                    sub = sub[sub['text'].apply(is_code_mixed)].reset_index(drop=True)
                    after = len(sub)
                    print(f"    ✓ Sheet '{sheet}': {after} code-mixed rows (filtered {before - after} pure Telugu)")
                    frames.append(sub)
                else:
                    print(f"    ⚠ Sheet '{sheet}': Skipped (cols={list(df.columns)})")
        except Exception as e:
            print(f"    ✗ Error: {e}")

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=['text', 'label', 'label_int'])

def load_badwords_as_training_data():
    toxic_words = []

    badwords_path = DATA_DIR / 'telugu_badwords.txt'
    if badwords_path.exists():
        with open(badwords_path, 'r', encoding='utf-8') as f:
            for line in f:
                word = line.strip()
                if word:
                    toxic_words.append(word)
        print(f"  ✓ Loaded {len(toxic_words)} words from telugu_badwords.txt")

    secure_path = DATA_DIR / 'secure_words.bin'
    secure_count = 0
    if secure_path.exists():
        with open(secure_path, 'rb') as f:
            decoded = base64.b64decode(f.read()).decode('utf-8')
            for line in decoded.splitlines():
                word = line.strip()
                if word and word not in toxic_words:
                    toxic_words.append(word)
                    secure_count += 1
        print(f"  ✓ Loaded {secure_count} additional words from secure_words.bin")

    emoji_path = DATA_DIR / 'bad_emojis.txt'
    emoji_count = 0
    if emoji_path.exists():
        with open(emoji_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith('#'):
                    toxic_words.append(line)
                    emoji_count += 1
        print(f"  ✓ Loaded {emoji_count} offensive emojis from bad_emojis.txt")

    if not toxic_words:
        return pd.DataFrame(columns=['text', 'label', 'label_int'])

    toxic_templates = [
        '{word}', 'you are a {word}', '{word} ga unnav', 'enti ra {word}',
        'orey {word}', 'nuvvu {word}', '{word} fellow', 'this {word}',
    ]
    safe_phrases = [
        'good morning everyone', 'nice video', 'great content bro',
        'keep it up', 'super ga undi', 'chala bagundi',
        'love this', 'awesome work', 'thank you for sharing',
        'very helpful', 'bagundi', 'nice one', 'well done',
        'interesting topic', 'manchi video', 'super explanation',
        'thanks for this', 'really useful', 'good job',
        'happy birthday', 'congratulations', 'best wishes',
        'nice song', 'beautiful', 'amazing performance',
        'very informative', 'subscribed', 'waiting for next video',
        'loved it', 'manchi content', 'edo oka roju',
        'nenu chala happy', 'meeru bagunnara', 'thanks anna',
        'thanks akka', 'super bro', 'nice edit',
        'first comment', 'who is watching in 2024',
        'please make more videos', 'this helped me a lot',
        'I learned something new', 'great tutorial', 'perfect',
    ]

    toxic_rows = []
    for word in toxic_words:
        templates = random.sample(toxic_templates, min(3, len(toxic_templates)))
        for template in templates:
            toxic_rows.append({'text': template.format(word=word), 'label': 'hate', 'label_int': 1})

    safe_rows = [{'text': safe_phrases[i % len(safe_phrases)], 'label': 'not-hate', 'label_int': 0} for i in range(len(toxic_rows))]

    print(f"  ✓ Generated {len(toxic_rows)} toxic + {len(safe_rows)} safe examples from bad words/emojis")
    return pd.DataFrame(toxic_rows + safe_rows)

# ── Load Data ─────────────────────────────────────────────────────────────────
print('\n📊 Loading training data...')
train_files = [f for f in DATA_DIR.iterdir() if 'training_data' in f.name.lower() and f.suffix in ['.xlsx', '.xls', '.csv']]

if not train_files:
    print('❌ No training file found! Make sure you uploaded training_data_telugu-hate.xlsx')
else:
    all_data = load_data(train_files)

    print('\n📝 Loading bad words/emojis...')
    badwords_data = load_badwords_as_training_data()
    if not badwords_data.empty:
        all_data = pd.concat([all_data, badwords_data], ignore_index=True)

    len_before = len(all_data)
    all_data = all_data.drop_duplicates(subset='text')
    print(f'  Deduplicated: {len_before} → {len(all_data)}')

    # Stratified split
    train_df, test_df = train_test_split(all_data, test_size=0.10, random_state=42, stratify=all_data['label_int'])
    print(f'\n📈 Train={len(train_df)} | Test={len(test_df)}')
    print(f'   Train classes: {train_df["label_int"].value_counts().to_dict()}')
    print(f'   Test classes:  {test_df["label_int"].value_counts().to_dict()}')

    # ── Tokenize ──────────────────────────────────────────────────────────────
    print(f'\n🔤 Loading tokenizer & model: {BASE_MODEL}')
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL, num_labels=2, ignore_mismatched_sizes=True,
        problem_type='single_label_classification'
    )
    print('✅ Model loaded')

    class CommentDataset(TorchDataset):
        def __init__(self, texts, labels):
            self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors='pt')
            self.labels = labels
        def __len__(self): return len(self.labels)
        def __getitem__(self, idx):
            item = {k: v[idx] for k, v in self.encodings.items()}
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
            return item

    print('Tokenizing...')
    train_dataset = CommentDataset(train_df['text'].tolist(), train_df['label_int'].tolist())
    test_dataset = CommentDataset(test_df['text'].tolist(), test_df['label_int'].tolist())

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {
            'accuracy': accuracy_score(labels, preds),
            'f1': f1_score(labels, preds, zero_division=0),
            'precision': precision_score(labels, preds, zero_division=0),
            'recall': recall_score(labels, preds, zero_division=0),
        }

    # ── Train ─────────────────────────────────────────────────────────────────
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'\n🚀 Training on: {device.upper()}')

    OUTPUT_DIR.mkdir(exist_ok=True)
    batch_size = 16 if device == 'cuda' else 8
    total_steps = (len(train_dataset) // batch_size) * EPOCHS
    warmup_steps = int(total_steps * 0.1)

    training_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=64,
        learning_rate=LEARNING_RATE,
        warmup_steps=warmup_steps,
        weight_decay=0.05,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1',
        logging_steps=25,
        report_to='none',
        fp16=(device == 'cuda'),
    )

    trainer = Trainer(
        model=model, args=training_args,
        train_dataset=train_dataset, eval_dataset=test_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    print('⏳ Training started...')
    # Automatically resume from the latest checkpoint if it exists in OUTPUT_DIR
    checkpoint_list = list(OUTPUT_DIR.glob('checkpoint-*'))
    resume_from_checkpoint = True if checkpoint_list else False
    if resume_from_checkpoint:
        print(f'🔄 Resuming from the latest checkpoint in {OUTPUT_DIR}')
    
    trainer.train(resume_from_checkpoint=resume_from_checkpoint)

    # ── Evaluate ──────────────────────────────────────────────────────────────
    print('\n📊 Evaluating...')
    results = trainer.evaluate()
    print(f"\n{'='*50}")
    print('🏆 FINAL RESULTS:')
    print(f"  Accuracy:  {results.get('eval_accuracy', 0)*100:.2f}%")
    print(f"  F1 Score:  {results.get('eval_f1', 0):.4f}")
    print(f"  Precision: {results.get('eval_precision', 0):.4f}")
    print(f"  Recall:    {results.get('eval_recall', 0):.4f}")
    print(f"{'='*50}")

    # ── Save ──────────────────────────────────────────────────────────────────
    trainer.save_model(str(OUTPUT_DIR))
    tokenizer.save_pretrained(str(OUTPUT_DIR))
    with open(OUTPUT_DIR / 'eval_results.json', 'w') as f:
        json.dump(results, f, indent=2)
    print(f'\n✅ Model saved to: {OUTPUT_DIR}')

## Step 4: Download the Trained Model

Run this cell to download the trained model as a **zip file**. Then extract it into your `backend/model_output/` folder on your PC.

In [ ]:
import shutil
from google.colab import files

# Zip the model
print('📦 Zipping model files...')
shutil.make_archive('comment_guard_model', 'zip', 'model_output')
print('✅ Zip created!')

# Download
print('⬇️ Starting download...')
files.download('comment_guard_model.zip')
print('\n📋 After download, extract the zip contents into your backend/model_output/ folder.')